# Extracción de Características con TSFEL

## 1. Importar librerías

In [ ]:
# Importar librerías necesarias
import os
import json
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import butter, filtfilt

# TSFEL para extracción de características
import tsfel

warnings.filterwarnings('ignore')

## 2. Parámetros del experimento

In [ ]:
# Parámetros
FPS         = 29.97
VENTANA     = 90 # 3 segundos
OVERLAP     = 45 # 50% overlap
ANGULOS     = ['Rodilla_I', 'Cadera_I', 'Tronco', 'Tobillo_I']
SUJETOS     = ['P1', 'P2', 'P3', 'P4', 'P5']
CONDICIONES = ['Inicial', 'Fatiga']
BASE        = Path('..')
OUTPUTS     = BASE / 'outputs'

np.random.seed(42)

print(f'Ventana: {VENTANA} frames ({VENTANA/FPS:.1f} segundos)')
print(f'Overlap: {OVERLAP} frames ({OVERLAP/FPS:.1f} segundos)')
print(f'Ángulos: {ANGULOS}')

## 3. Funciones auxiliares (del notebook anterior)

In [ ]:
# Funciones de preprocesamiento (copiadas del notebook anterior)

def extraer_punto(keypoints, indice):
    # Extrae coordenadas (x, y, confianza) de un keypoint
    b = indice * 3
    return (keypoints[b], keypoints[b+1], keypoints[b+2])


def calcular_angulo(p1, p2, p3, umbral=0.1):
    # Calcula el ángulo en p2 formado por p1-p2-p3
    if any(p[2] < umbral for p in [p1, p2, p3]):
        return None
    v1 = (p1[0]-p2[0], p1[1]-p2[1])
    v2 = (p3[0]-p2[0], p3[1]-p2[1])
    dot = v1[0]*v2[0] + v1[1]*v2[1]
    m1  = math.sqrt(v1[0]**2 + v1[1]**2)
    m2  = math.sqrt(v2[0]**2 + v2[1]**2)
    if m1 == 0 or m2 == 0:
        return None
    return round(math.degrees(math.acos(max(-1.0, min(1.0, dot/(m1*m2))))), 2)


def calcular_angulo_tronco(hombro, cadera, umbral=0.1):
    # Calcula la inclinación del tronco respecto a la vertical
    if hombro[2] < umbral or cadera[2] < umbral:
        return None
    vx  = hombro[0] - cadera[0]
    vy  = hombro[1] - cadera[1]
    mag = math.sqrt(vx**2 + vy**2)
    if mag == 0:
        return None
    dot = (vx*0 + vy*(-1)) / mag
    return round(math.degrees(math.acos(max(-1.0, min(1.0, dot)))), 2)


def seleccionar_ciclista(personas, ancho_frame=1920):
    # Selecciona la persona más cercana al centro horizontal
    if len(personas) == 1:
        return personas[0]['pose_keypoints_2d']
    centro_frame = ancho_frame / 2
    mejor_idx, menor_dist = 0, float('inf')
    for idx, persona in enumerate(personas):
        kp = persona['pose_keypoints_2d']
        puntos_x = [kp[i*3] for i in [1,2,5,8,9,12]
                    if kp[i*3+2] > 0.1 and kp[i*3] > 0]
        if not puntos_x:
            continue
        dist = abs(sum(puntos_x)/len(puntos_x) - centro_frame)
        if dist < menor_dist:
            menor_dist = dist
            mejor_idx  = idx
    return personas[mejor_idx]['pose_keypoints_2d']


def butter_lowpass(serie, fc=6.0, fps=29.97, orden=4):
    # Aplica filtro Butterworth de paso bajo
    nyquist = fps / 2.0
    b, a    = butter(orden, fc/nyquist, btype='low', analog=False)
    serie_interp = serie.interpolate(method='linear', limit_direction='both')
    nan_mask = serie.isna()
    if serie_interp.notna().sum() < 15:
        return serie.values
    filtrada = filtfilt(b, a, serie_interp.values)
    filtrada[nan_mask] = np.nan
    return filtrada


def cargar_serie(sujeto, condicion, outputs_dir, fps=29.97):
    # Carga los JSON y calcula ángulos filtrados
    json_dir = outputs_dir / f'{sujeto}_{condicion}' / 'json'
    archivos = sorted([f for f in os.listdir(json_dir)
                       if f.endswith('_keypoints.json')])
    
    ancho_frame = 1920
    for nombre in archivos[:10]:
        try:
            with open(json_dir / nombre) as f:
                d = json.load(f)
            if d.get('people'):
                kp = d['people'][0]['pose_keypoints_2d']
                xs = [kp[i*3] for i in range(25) if kp[i*3] > 0]
                if xs:
                    ancho_frame = max(xs) * 1.1
                    break
        except Exception:
            continue
    
    filas = []
    for nombre in archivos:
        try:
            with open(json_dir / nombre) as f:
                datos = json.load(f)
        except Exception:
            continue
        
        if not datos.get('people'):
            filas.append({'Rodilla_I': None, 'Cadera_I': None,
                          'Tronco': None, 'Tobillo_I': None})
            continue
        
        kp = seleccionar_ciclista(datos['people'], ancho_frame)
        p  = {n: extraer_punto(kp, n) for n in range(25)}
        
        filas.append({
            'Rodilla_I': calcular_angulo(p[12], p[13], p[14]),
            'Cadera_I': calcular_angulo(p[1],  p[12], p[13]),
            'Tronco': calcular_angulo_tronco(p[5], p[12]),
            'Tobillo_I': calcular_angulo(p[13], p[14], p[19]),
        })
    
    df = pd.DataFrame(filas)
    for angulo in ANGULOS:
        df[angulo] = butter_lowpass(df[angulo], fps=fps)
    
    return df

print(' Funciones definidas')

## 4. Cargar datos

In [ ]:
print('Cargando datos...')
datos_crudos = {}

for sujeto in SUJETOS:
    for condicion in CONDICIONES:
        key = f'{sujeto}_{condicion}'
        df = cargar_serie(sujeto, condicion, OUTPUTS, FPS)
        datos_crudos[key] = df
        frames_validos = df[ANGULOS].dropna().shape[0]
        print(f'  {key}: {len(df)} frames, {frames_validos} válidos')

print('\n Datos cargados')

## 5. Normalización Z-score

In [ ]:
# Calcular estadísticas globales
todos = pd.concat(list(datos_crudos.values()), ignore_index=True)
MEDIA_GLOBAL = {}
STD_GLOBAL   = {}

for angulo in ANGULOS:
    MEDIA_GLOBAL[angulo] = todos[angulo].dropna().mean()
    STD_GLOBAL[angulo]   = todos[angulo].dropna().std()
    print(f'{angulo}: media={MEDIA_GLOBAL[angulo]:.2f}, std={STD_GLOBAL[angulo]:.2f}')

def normalizar(df):
    df_norm = df.copy()
    for angulo in ANGULOS:
        df_norm[angulo] = (df[angulo] - MEDIA_GLOBAL[angulo]) / STD_GLOBAL[angulo]
        df_norm[angulo] = df_norm[angulo].fillna(0.0)
    return df_norm

datos_norm = {key: normalizar(df) for key, df in datos_crudos.items()}
print('\n Normalización aplicada')

## 6. Configurar TSFEL

In [ ]:
# Obtener todas las características disponibles
cfg = tsfel.get_features_by_domain()

# Seleccionar características relevantes para biomecánica
features_selected = {
    'statistical': [
        'Mean',                    # Media
        'Standard deviation',       # Desviación estándar
        'Variance',                # Varianza
        'Max',                     # Máximo
        'Min',                     # Mínimo
        'Root mean square',        # RMS (energía)
        'Interquartile range',     # Rango intercuartil
    ],
    'temporal': [
        'Zero crossing rate',      # Tasa de cruces por cero
        'Slope',                   # Pendiente
        'Mean abs diff',           # Diferencia absoluta media
    ],
    'spectral': [
        'FFT mean coefficient',    # Coeficiente medio de FFT
        'Fundamental frequency',   # Frecuencia fundamental
        'Max power spectrum',      # Potencia espectral máxima
        'Spectral centroid',       # Centroide espectral
        'Spectral entropy',        # Entropía espectral
    ]
}

# Crear configuración personalizada
cfg_custom = {}
for domain, feat_list in features_selected.items():
    if domain not in cfg_custom:
        cfg_custom[domain] = {}
    for feat_name in feat_list:
        if feat_name in cfg[domain]:
            cfg_custom[domain][feat_name] = cfg[domain][feat_name]

print('Características seleccionadas:')
for domain, feats in cfg_custom.items():
    print(f'  {domain}: {len(feats)} características')

## 7. Extracción de características con TSFEL

In [ ]:
def generar_ventanas_con_features(df_norm, etiqueta, sujeto, ventana=90, overlap=45):
    """
    Divide la serie en ventanas y extrae características TSFEL.
    Retorna lista de diccionarios con features y metadata.
    """
    valores = df_norm[ANGULOS].values
    paso = ventana - overlap
    muestras = []
    
    for inicio in range(0, len(valores) - ventana + 1, paso):
        fragmento = valores[inicio: inicio + ventana]
        
        # Verificar calidad: al menos 70% válido
        if np.isnan(fragmento).mean() < 0.3:
            fragmento_clean = np.nan_to_num(fragmento, nan=0.0)
            
            features_dict = {'sujeto': sujeto, 'y': etiqueta}
            
            # Extraer características de cada ángulo
            for idx, angulo in enumerate(ANGULOS):
                serie = fragmento_clean[:, idx]
                
                try:
                    df_temp = pd.DataFrame({angulo: serie})
                    feats = tsfel.time_series_features_extractor(
                        cfg_custom, df_temp, fs=FPS, verbose=0
                    )
                    
                    # Renombrar con el nombre del ángulo
                    for col in feats.columns:
                        feature_name = f'{angulo}_{col}'
                        features_dict[feature_name] = feats[col].values[0]
                
                except Exception as e:
                    print(f'Warning: Error en {angulo}: {e}')
                    continue
            
            muestras.append(features_dict)
    
    return muestras

In [ ]:
# Generar dataset con características
print('Extrayendo características...')
print('Esto puede tardar varios minutos...\n')

todas_las_muestras = []

for sujeto in SUJETOS:
    for condicion, etiqueta in zip(CONDICIONES, [0, 1]):
        key = f'{sujeto}_{condicion}'
        print(f'Procesando {key}...')
        
        muestras = generar_ventanas_con_features(
            datos_norm[key], etiqueta, sujeto, VENTANA, OVERLAP
        )
        
        todas_las_muestras.extend(muestras)
        print(f'{len(muestras)} ventanas (etiqueta={etiqueta})\n')

print(f'Total: {len(todas_las_muestras)} ventanas procesadas')

## 8. Crear DataFrame y guardar

In [ ]:
# Convertir a DataFrame
df_features = pd.DataFrame(todas_las_muestras)

# Información del dataset
feature_cols = [col for col in df_features.columns if col not in ['sujeto', 'y']]
print(f'Total de características: {len(feature_cols)}')
print(f'Características por ángulo: ~{len(feature_cols) // 4}')

# Balance de clases
n_inicial = (df_features['y'] == 0).sum()
n_fatiga  = (df_features['y'] == 1).sum()
print(f'\nBalance de clases:')
print(f'Inicial (0): {n_inicial} ventanas')
print(f'Fatiga (1):  {n_fatiga} ventanas')

# Mostrar primeras filas
print('\nPrimeras 5 filas:')
df_features.head()

In [ ]:
# Guardar dataset de características
df_features.to_csv(OUTPUTS / 'data' / 'features_tsfel.csv', index=False)
print(f'Dataset guardado en: outputs/data/features_tsfel.csv')

# Guardar lista de características para referencia
df_feature_list = pd.DataFrame({'feature': feature_cols})
df_feature_list.to_csv(OUTPUTS / 'data' / 'feature_names.csv', index=False)
print(f'Lista de características guardada')